<a href="https://colab.research.google.com/github/acoiman/pdt/blob/main/rr_respiratory_mortality/notebooks/06.RR_Respiratory_Mortality_Spatial_Cross_Correlation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🔄 Spatial Cross-Correlation and Bivariate Spatial Analysis Between Predicted Asthma Mortality Rate and RR of Respiratory Mortality — Argentina (2022)

In this notebook, we conduct a bivariate spatial analysis to examine the geographic relationship between the predicted asthma mortality rate—derived from a Random Forest model—and the central bound of the respiratory mortality RR across departments in Argentina for 2022. For this analysis, we focus on the central bound of respiratory mortality RR as its spatial autocorrelation is consistent across all bounds.

Rather than analyzing each variable independently, bivariate spatial analysis evaluates whether high (or low) values of one variable tend to cluster spatially near high (or low) values of another, thereby revealing cross-variable geographic dependencies that standard correlation methods cannot capture.

The workflow follows four methodological steps: i)constructing a Queen spatial weights matrix to represent neighborhood structure; ii)computing the bivariate global Moran’s I statistic to quantify overall spatial cross-correlation; iii) deriving local indicators of spatial association (LISA) to identify department-level cluster types; iv) mapping these cluster patterns;

## 🤖 Load libraries

The libraries required for the analysis will be loaded

In [ ]:
# dataframe libraries
import pandas as pd

# Import NumPy for numerical and mathematical operations
import numpy as np

# geospatial libraries
import geopandas as gpd

# spatial modeling libraries
from spreg import OLS  # PySAL OLS regression
from spreg import ML_Lag, ML_Error

# geostatistical libraries
import pysal
from pysal.lib import weights
import libpysal
from libpysal.weights import Queen, KNN, W
from libpysal.weights.spatial_lag import lag_spatial
import esda
from esda.moran import Moran
from esda.moran import Moran_Local
from esda.moran import Moran_BV, Moran_Local_BV

# plot libraries
import mapclassify
from mpl_toolkits.axes_grid1 import make_axes_locatable  # for colourbar sizing
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
import matplotlib.patches as mpatches
from shapely.geometry import box
import matplotlib.lines as mlines
from matplotlib.colors import Normalize, ListedColormap
import seaborn as sns


# other libraries
from adjustText import adjust_text
from IPython.display import Image
import os
from itables import init_notebook_mode, show

In [ ]:
# change directory to work folder (at the begining docker container enter into /home/jovyan/)
%cd work

In [ ]:
# Set the PROJ_LIB path
os.environ['PROJ_LIB'] = "/opt/conda/envs/gds/share/proj"

In [ ]:
 # Copy-on-Write
 pd.options.mode.copy_on_write = True

##🗂 Load and prepare the dataset

In [ ]:
# open gdf data
gdf = gpd.read_file("pdt/rr_respiratory_mortality/data/gpkg/gdf_rrrm.gpkg")

In [ ]:
# visualize geodataframe
init_notebook_mode(all_interactive=True)
gdf

In [ ]:
# select 'IDDPTO', 'AMR_PRED_2022' 'RRRM_2022_L' 'RRRM_2022_C', 'RRRM_2022_U', 'geometry' columns
gdf_ca_rrrm = gdf[[ 'IDDPTO', 'AMR_PRED_2022', 'RRRM_2022_L', 'RRRM_2022_C', 'RRRM_2022_U', 'geometry']]

In [ ]:
# visualize gdf
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm

In [ ]:
# original length of data
len(gdf_ca_rrrm)

In [ ]:
# drop all rows containing NaN for all rows
gdf_ca_rrrm = gdf_ca_rrrm.dropna().reset_index(drop=True)

In [ ]:
# length of data after dropping NaN
len(gdf_ca_rrrm)

In [ ]:
# visualize gdf
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm

## 🏗️ Construct the Queen contiguity matrix



In [ ]:
# project gdf_ca_rrrm to 5347
gdf_ca_rrrm_pj= gdf_ca_rrrm.to_crs(5347)

In [ ]:
# Calculate Queen contiguity matrix
w_queen = weights.contiguity.Queen.from_dataframe(gdf_ca_rrrm_pj, use_index=True)

In [ ]:
# Check for isolated departments (no neighbors)
print("Number of islands (Queen):", len(w_queen.islands))

In [ ]:
# transform raw weights into row-standardized
w_queen.transform = 'R'

### Mapping Queen contiguity matrix

In [ ]:
# select all NaN rows for gdf
gdf_nan = gdf[gdf.isna().any(axis=1)]

In [ ]:
# from gdf_nan select IDDPTO and geometry
gdf_nan = gdf_nan[['IDDPTO', 'geometry']]

In [ ]:
# reproject gdf_nan
gdf_nan_pj = gdf_nan.to_crs(5347)

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(6, 8))

# Queen contiguity matrix
gdf_ca_rrrm_pj.plot(ax=ax)
w_queen.plot(
    gdf_ca_rrrm_pj,
    edge_kws=dict(linewidth=1, color="orangered"),
    node_kws=dict(marker="*"),
    ax=ax,
)
# NaN departments, where asthma mortality did not ocurr
gdf_nan_pj.plot(ax=ax, color="none", edgecolor="black", linewidth=0.15)
ax.set_axis_off()
ax.set_title("Queen contiguity matrix")

f.tight_layout()
plt.show()

## ⇄ Bivariate Global Moran's I

The **Bivariate Global Moran's I** extends the classic univariate Moran's I to measure the degree to which high (or low) values of one variable at a location $i$ are spatially associated with high (or low) values of a *different* variable at neighboring locations $j$. Formally, it is defined as:

$$
I_B = \frac{\sum_i \sum_j w_{ij} z_{x,i} z_{y,j}}{\sum_i z_{x,i}^2}
$$

where $z_{x,i}$​ and $z_{y,j}$​ are the standardised (z-scored) values of variables *x* and *y* respectively, and $w_{ij}$ ​ are the row-standardised spatial weights. A **positive** $I_B$​ indicates that departments with high predicted asthma mortality tend to be surrounded by departments with high RR of respiratory mortality (spatial co-clustering), while a **negative** value would suggest a spatial divergence between the two variables. Unlike a standard Pearson correlation, this statistic is explicitly spatial — it captures cross-variable dependence *across space*, not just within the same unit.[1][2]



In [ ]:
# Extract variable arrays
# x: focal variable — predicted asthma mortality rate (AMR_PRED_2022)
x = gdf_ca_rrrm['AMR_PRED_2022'].values

# y: lagged variable — central attributable RR of respiratory mortality (RRRM_2022_C)
y = gdf_ca_rrrm['RRRM_2022_U'].values

In [ ]:
# Compute Bivariate Global Moran's I
# Moran_BV(x, y, w) computes I_B = cross-correlation of x_i with
# the spatial lag of y at neighbouring locations j.
# permutations=999 builds the null distribution via random reassignment.
bv_moran = Moran_BV(x, y, w_queen, permutations=999)

In [ ]:
# Print results

# EI_sim is an array of 999 simulated I_B values → mean = average simulated I
expected_I = bv_moran.EI_sim.mean()

# seI_sim is an array → mean gives the average std dev across permutations
se_I = bv_moran.seI_sim.mean()

# Z-score: how many SDs the observed I sits above the simulated mean
z_score = (bv_moran.I - expected_I) / se_I

print("── Bivariate Global Moran's I Results ──────────────────────────")
print(f"  Observed I_B                    : {bv_moran.I:.4f}")
print(f"  Expected I_B (mean of sim)      : {expected_I:.4f}")
print(f"  Std. Dev. (mean of seI_sim)     : {se_I:.4f}")
print(f"  Z-score                         : {z_score:.4f}")
print(f"  Pseudo p-value (permutations)   : {bv_moran.p_sim:.4f}")
print(f"  P-value (normal approximation)  : {bv_moran.p_z_sim:.4f}")
print(f"  Permutations run                : 999")

The Bivariate Global Moran's I between predicted asthma mortality rate and the spatially lagged central attributable relative risk (RR) of respiratory mortality yields $I_B$​  = -0.0962. While close to the expected value under spatial randomness (−0.0041), the conditional permutation test yields a pseudo p-value of 0.0400, which is below the conventional α = 0.05 threshold (Z-score ≈ −1.59, normal approximation p-value = 0.0556). This indicates a statistically significant, weak negative global spatial cross-correlation between these two variables across Argentine departments, meaning high values of one variable tend to be spatially adjacent to lower values of the other

## 📍 Compute Bivariate Local Moran's I (LISA)

Local Indicators of Spatial Association (LISA) decompose the global spatial autocorrelation into department-level contributions, revealing where spatial cross-correlation is concentrated. The Bivariate Local Moran's I for unit i is defined as:

$$
I_{B,i} = z_{x,i} \sum_{j} w_{ij} z_{y,j}
$$

where $z_{x,i}$ is the standardised predicted asthma mortality rate at unit *i*, and $\sum_j w_{ij} z_{y,j}$​ is the spatially lagged standardised attributable case rate at *j*'s neighbours. Each department receives a local $I_{B,i}$​ score and, through a conditional permutation test, a pseudo p-value indicating whether its local cross-correlation. Although the bivariate global Moran’s I is very weak, we can still compute the bivariate local Moran’s I (LISA), as some departments may exhibit statistically significant local spatial associations.[3]
*
In this section, we will perform the following steps:

* Calculate the bivariate local Moran’s I (LISA).
* Filter departments by excluding islands and those with non-significant LISA values.
* Assign quadrant labels to significant departments and classify them into HH, HL, LH, and LL clusters.
* Extract neighboring departments for each cluster type.

### Calculate the bivariate local Moran’s I (LISA)

In [ ]:
# x: focal variable (Predicted asthma mortality rate CA_2022_PRED)
# y: lagged variable (RR of respiratory mortality central bound-RRRM_2022_C)
# permutations=999: conditional permutation test for each of the 170 units
with np.errstate(divide='ignore', invalid='ignore'):
    lisa = Moran_Local_BV(x, y, w_queen, permutations=9999)

In [ ]:
# Store local I_B (value of Moran’s I) values in the GeoDataFrame
# lisa.Is: array of 170 local bivariate Moran's I scores (one per department)
gdf_ca_rrrm_pj['LOCAL_I'] = lisa.Is

In [ ]:
# Store pseudo p-values
# lisa.p_sim: array of 170 pseudo p-values from conditional permutation test
# p-value based on permutations (one-sided) null: spatial randomness
# alternative: the observed Ii is further away or extreme from the median of simulated values.
gdf_ca_rrrm_pj['P_SIM'] = lisa.p_sim

In [ ]:
#  move geometry column to the end
gdf_ca_rrrm_pj = gdf_ca_rrrm_pj[[col for col in gdf_ca_rrrm_pj.columns if col != 'geometry'] + ['geometry']]

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm_pj

### Filter departments by excluding islands and those with non-significant LISA p-values

Spatial units without contiguous neighbors (islands) will be excluded from the local spatial autocorrelation analysis, as the absence of spatial lag prevents meaningful computation of Local Moran’s statistics. These units wiil be retained in the geographic dataset but won't be interpreted in the clustering results. We also exclude departments with non-significant LISA p-values.

In [ ]:
# Create a column named "IS_ISLAND" in gdf_ca_rrrm_pj,
# based on w_queen.cardinalities set 1 if the department is an island, otherwise 0 if it is not an island
gdf_ca_rrrm_pj['IS_ISLAND'] = (gdf_ca_rrrm_pj.index.map(w_queen.cardinalities) == 0).astype(int)

In [ ]:
#  move geometry column to the end
gdf_ca_rrrm_pj = gdf_ca_rrrm_pj[[col for col in gdf_ca_rrrm_pj.columns if col != 'geometry'] + ['geometry']]

In [ ]:
# # visualize dataframe
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm_pj

In [ ]:
# Create a column named "SIGNIFICANT" in gdf_ca_rrrm_pj, set 1 if 1 P_SIM < 0.05 and IS_ISLAND == 0, otherwise set 0
gdf_ca_rrrm_pj['SIGNIFICANT'] = ((gdf_ca_rrrm_pj['P_SIM'] < 0.05) & (gdf_ca_rrrm_pj['IS_ISLAND'] == 0)).astype(int)

In [ ]:
#  move geometry column to the end
gdf_ca_rrrm_pj = gdf_ca_rrrm_pj[[col for col in gdf_ca_rrrm_pj.columns if col != 'geometry'] + ['geometry']]

In [ ]:
# # visualize dataframe
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm_pj

### Assign quadrant labels to significant departments

In [ ]:
# create a column to store quadrant codes
# lisa.q: integer codes → 1=HH, 2=LH, 3=LL, 4=HL
gdf_ca_rrrm_pj['QUADRANT'] = lisa.q

In [ ]:
#  move geometry column to the end
gdf_ca_rrrm_pj = gdf_ca_rrrm_pj[[col for col in gdf_ca_rrrm_pj.columns if col != 'geometry'] + ['geometry']]

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm_pj

In [ ]:
# Label departments based on their p-value significance level and quadrant
# e.g. 1S means the department is in quadrant 1 and is significant
sig_quad_labels = {1: 'S1', 2: 'S2', 3: 'S3', 4: 'S4'}

In [ ]:
gdf_ca_rrrm_pj['SIG_QUAD'] = gdf_ca_rrrm_pj.apply(
    lambda row: sig_quad_labels[row['QUADRANT']] if row['SIGNIFICANT'] == 1 else 'NSC', # NSC = No Significant Cluster
    axis=1
)

In [ ]:
#  move geometry column to the end
gdf_ca_rrrm_pj = gdf_ca_rrrm_pj[[col for col in gdf_ca_rrrm_pj.columns if col != 'geometry'] + ['geometry']]

In [ ]:
# Display the updated GeoDataFrame with the new column
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm_pj

### Classify neighboring departments for each cluster type

In this step, departments will be classified into four **cluster types**​: **HH** (high mortality surrounded by high RR of respiratory mortality), **LL** (low surrounded by low), **HL** (high surrounded by low), and **LH** (low surrounded by high) — with only neighbouring and statistically significant units (typically p < 0.05) retained for mapping.

From an epidemiological perspective, we will prioritize spatial cluster types by their potential to identify high-risk areas:

$HH > HL > LH > LL$ [4]

1. High-High (HH) - "Hot Spots" (Highest Priority):
2. High-Low (HL) - "Spatial Outliers" (High Priority)
3. Low-High (LH) - "Spatial Outliers" (Medium Priority)
4. Low-Low (LL) - "Cold Spots" (Lowest Priority

We will apply the following equivalence accodding to LISA quadrant diagram:

* HH > HL > LH > LL  (epidemiological priority)
* (HH=1) > (HL=4) > (LH=2) > (LL=3) (Pysal priority)
* S1 > S4 > S2 > S3  (our priority equivalence for SIG_QUAD column)

In [ ]:
# display LISA quadrants diagram
print("LISA quadrants diagram")
Image(filename='pdt/rr_respiratory_mortality/data/images/LISA-quadrants-diagram.jpg', width=600)

In [ ]:
# create a colulumn for cluster classes
gdf_ca_rrrm_pj['CLUSTERS'] = 'NO CLUSTER'

In [ ]:
#  PySAL Scheme: HH=1, LH=2, LL=3, HL=4
cluster_type_labels = {"S1": 'HH', "S2": 'LH', "S3": 'LL', "S4": 'HL'}

#### HH Classification (High predicted asthma mortality rate surrounded by High RR of respiratory mortality)

To classify departments as HH clusters, we will perform the following steps:

* Identify the indices of departments in Moran’s Quadrant 1 with statistically significant LISA p-values.
* Retrieve the Queen contiguity neighbors of these departments.
* Plot the resulting cluster to visually inspect for islands and any non-significant LISA departments.
* Filter the final set of clustered departments based on spatial contiguity and statistically significant LISA p-values.



In [ ]:
# get index of SIG_QUAD == S1 (significant LISA p-value in quadrant 1)
index_s1 = gdf_ca_rrrm_pj[gdf_ca_rrrm_pj['SIG_QUAD'] == "S1"].index

In [ ]:
# index to list
index_s1 = index_s1.to_list()

In [ ]:
# print index
index_s1

In [ ]:
# function to get neighbours from index/es list
def get_neighbouring(index_list):
  """
  index_list: list of index/es
  return: all_index_with_neighbors_flatten: list of all index/es with their neighbors
  """
  all_index_with_neighbors = []
  for idx in index_list:
    neighbors = w_queen.neighbors.get(idx, []) # Get neighbors, or an empty list if no neighbors (island)
    all_index_with_neighbors.append({idx: neighbors})

    # print with the last iteration
    if idx == index_list[-1]:
      print("Indexes of all_index and its neighbors:")
      print(all_index_with_neighbors)

  flattened_indices = set()
  for item_dict in all_index_with_neighbors:
    for key, values in item_dict.items():
        flattened_indices.add(key)
        for val in values:
            flattened_indices.add(val)
  all_index_with_neighbors_flatten= sorted(list(flattened_indices))
  print("all_index_with_neighbors_flatten: ", all_index_with_neighbors_flatten)
  return all_index_with_neighbors_flatten

In [ ]:
# apply the function to get index of  index_s1 neighbours
index_s1_neigh = get_neighbouring(index_s1)

**No HH (High predicted asthma mortality rate surrounded by High RR of respiratory mortality) cluster was identified**

#### HL Classification (High predicted asthma mortality rate surrounded by Low RR of respiratory mortality)

In [ ]:
# function to visually inspect the neighbouring
def plot_neighbouring(index):
  ax = index.plot(figsize=(6, 6))
  ax.axis('off')

  # Add index labels to each polygon
  for idx, row in index.iterrows():
    # Get the centroid of the polygon
    centroid = row.geometry.centroid
    # Annotate the polygon with its index
    ax.annotate(text=str(idx), xy=(centroid.x, centroid.y), ha='center', va='center', fontsize=8, color='black')

In [ ]:
# get index of SIG_QUAD == S4 (significant LISA p-value in quadrant 4)
index_s4 = gdf_ca_rrrm_pj[gdf_ca_rrrm_pj['SIG_QUAD'] == "S4"].index

In [ ]:
# instex to list
index_s4 = index_s4.to_list()

In [ ]:
# print index
index_s4

In [ ]:
# function to verify and exclude neighbouring index/es  that are included in a different neighbouring index/es list
def verify_exclude_neighbouring(current_index, previous_index):
  '''
  Verify and exclude neighbouring index/es are not included in a diiferent neighbouring index/es
  current_index: current list of index/es
  previous_index: previous list of index/es
  '''
  # Convert previous_index to a set for efficient lookup
  previous_index_set = set(previous_index)

  # Create a new list to store items that are not excluded
  new_index = []

  for item in current_index:
      if item in previous_index_set:
         print(f"Excluding item from current index: {item}")
      else:
         new_index.append(item)

  # Update current_index with the new list
  current_index = new_index

  # Print the updated current_index
  print(f"Updated current index: {current_index}")

  return current_index

In [ ]:
# apply the function to verify and/o exclude any index included in HH neighboring
index_s4 = verify_exclude_neighbouring(index_s4, index_s1_neigh)

In [ ]:
# print index_s4
index_s4

In [ ]:
# apply the function to get neighbours indexes of each items in index_s4  list
index_s4_neigh = get_neighbouring(index_s4)

In [ ]:
# apply the function to verify and/o exclude any index included in HH neighboring
index_s4_neigh = verify_exclude_neighbouring(index_s4_neigh, index_s1_neigh)

In [ ]:
# get data of items in index_s4_neigh list
S4 = gdf_ca_rrrm_pj.loc[index_s4_neigh]

In [ ]:
# move geometry column to the end
S4 = S4[[col for col in S4.columns if col != 'geometry'] + ['geometry']]

In [ ]:
# Visualize S4 dataframe
init_notebook_mode(all_interactive=True)
S4

In [ ]:
# plot S4 neighbouring
plot_neighbouring(S4)

We observe a contiguous cluster consistent with the Queen contiguity matrix; however, we will exclude indices 92 and 100 because they do not have statistically significant LISA p-values. We will also exclude index 101, as it becomes an island after removing index 100.

In [ ]:
# get list of indexes to drop
index_to_exclude_S4 = S4[S4['SIG_QUAD']=="NSC"].index.to_list()
# add index 101
index_to_exclude_S4.append(101)

In [ ]:
# verify indexes to exclude
index_to_exclude_S4

In [ ]:
# exclude indexes from index_s4_neigh
index_s4_neigh = [item for item in index_s4_neigh if item not in index_to_exclude_S4]

In [ ]:
# print index_s4_neigh
index_s4_neigh

In [ ]:
# reselect data and visualize the datafarme
S4 = gdf_ca_rrrm_pj.loc[index_s4_neigh]
S4 = S4[[col for col in S4.columns if col != 'geometry'] + ['geometry']]
init_notebook_mode(all_interactive=True)
S4

In [ ]:
# Replot S4 neighbouring
plot_neighbouring(S4)

In [ ]:
# set cluster class in gdf_ca_rrrm_pj dataset (CLUSTERS == HL)
for index in index_s4_neigh:
  gdf_ca_rrrm_pj.loc[index, 'CLUSTERS'] = 'HL'

In [ ]:
#  move geometry column to the end
gdf_ca_rrrm_pj = gdf_ca_rrrm_pj[[col for col in gdf_ca_rrrm_pj.columns if col != 'geometry'] + ['geometry']]

In [ ]:
# visiualize rows of HL cluster in
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm_pj.loc[index_s4_neigh]

##### Mapping HL cluster

In [ ]:
# Recover NaN departments (no recorded asthma mortality)
# These were excluded from the analysis but must appear on the map
# to preserve the full national geography of Argentina.
gdf_nan = gdf[gdf.isna().any(axis=1)][['IDDPTO', 'geometry']]

In [ ]:
# Reproject NaN departments to POSGAR 2007 (EPSG:5347) to match gdf
gdf_nan_pj = gdf_nan.to_crs(5347)

In [ ]:
# open gpkg with province and department name data
gdf_data= gpd.read_file("pdt/asthma_mortality/data/gpkg/data.gpkg")

In [ ]:
# select PROV IDDPTO and DPTO from gdf_data
gdf_data_prov_dep = gdf_data[['PROV', 'IDDPTO', 'DPTO']]

In [ ]:
# join gdf_ca_rrrm_pj and gdf_data_prov_dep by IDDPTO that only match n gdf_ca_rrrm_pj rows
gdf_ca_rrrm_pj = gdf_ca_rrrm_pj.merge(gdf_data_prov_dep, on='IDDPTO', how='left')

In [ ]:
#  move geometry column to the end
gdf_ca_rrrm_pj = gdf_ca_rrrm_pj[[col for col in gdf_ca_rrrm_pj.columns if col != 'geometry'] + ['geometry']]

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm_pj.head()

In [ ]:
# Filter data
HL_data = gdf_ca_rrrm_pj[gdf_ca_rrrm_pj['CLUSTERS'] == 'HL']
non_HL_data = gdf_ca_rrrm_pj[gdf_ca_rrrm_pj['CLUSTERS'] != 'HL']

# Set up figure and two subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 6), gridspec_kw={'wspace': 0.02})

# --- First Subplot (Map 1) ---
ax1 = axes[0]

# Plot non-HL departments as hollow with black border
non_HL_data.plot(
    ax=ax1,
    color='none',
    edgecolor='black',
    linewidth=0.15
)

# Plot NaN departments as hollow with black border
gdf_nan_pj.plot(
    ax=ax1,
    color='none',
    edgecolor='black',
    linewidth=0.15
)

# Plot HL departments in red with white borders
HL_data.plot(
    ax=ax1,
    color='#fdae61',
    edgecolor='black',
    linewidth=0.3
)
ax1.set_axis_off()

# --- Second Subplot (Map 2) ---
ax2 = axes[1]

# Add the hollow black rectangle around HL_data on ax1
if not HL_data.empty:
    minx, miny, maxx, maxy = HL_data.union_all().bounds
    rect_patch = mpatches.Rectangle((minx, miny), maxx - minx, maxy - miny,
                                    facecolor='none', edgecolor='black', linewidth=2, linestyle='--', zorder=5)
    ax1.add_patch(rect_patch)

    # Add a black arrow pointing from the rectangle to the second subplot
    # The arrow starts from the right-middle of the rectangle in ax1's data coordinates
    # and points to the left-middle of ax2's axes coordinates.
    ax1.annotate('',
                 xy=(0, 0.67), # Target point (left edge, center) on ax2
                 xycoords=ax2.transAxes, # Interpret xy in ax2's axes coordinates
                 xytext=(maxx, miny + (maxy - miny) / 2), # Start point on ax1 (right-middle of rectangle)
                 textcoords=ax1.transData, # Interpret xytext in ax1's data coordinates
                 arrowprops=dict(facecolor='black', edgecolor='black',
                                 width=0.1, headwidth=6, headlength=8))

# --- Second Subplot (Map 2) ---
# ax2 = axes[1]

# Plot only HL departments in red with white borders
HL_data.plot(
    ax=ax2,
    color='#fdae61',
    edgecolor='black',
    linewidth=0.3
)

# Add IDDPTO labels to the center of each HL polygon
for idx, row in HL_data.iterrows():
    centroid = row.geometry.centroid
    ax2.annotate(text=str(row['IDDPTO']), xy=(centroid.x, centroid.y), ha='center', va='center', fontsize=7, color='black')

ax2.set_axis_off()

# --- General Title and Shared Legend ---
fig.suptitle(
    "High predicted asthma mortality rate surrounded by \nLow RR of respiratory mortality",
    fontsize=14, fontweight='bold', y=0.95 # Adjust y for spacing
)

# Create a custom legend patch for HL
hl_patch = mpatches.Patch(facecolor='#fdae61', edgecolor='white', label='High-Low (HL) - "Spatial Outliers"')
fig.legend(
    handles=[hl_patch],
    loc='lower center',
    bbox_to_anchor=(0.5, 0.07), # Position below both plots
    ncol=1,
    frameon=False,
    fontsize=10
)

#plt.tight_layout(rect=[0, 0.05, 1, 0.98]) # Adjust layout to make space for the legend
plt.show()

In [ ]:
# display IDDPTO, PROV and DPTO HL cluster
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm_pj.loc[index_s4_neigh, ['IDDPTO', 'PROV', 'DPTO']]

#### LH Classification (Low predicted asthma mortality rate surrounded by High RR of respiratory mortality)

In [ ]:
# get index of SIG_QUAD == S2 (significant LISA p-value in quadrant 2)
index_s2 = gdf_ca_rrrm_pj[gdf_ca_rrrm_pj['SIG_QUAD'] == "S2"].index

In [ ]:
# instex to list
index_s2 = index_s2.to_list()

In [ ]:
# print index
index_s2

In [ ]:
# apply the function to verify and/o exclude any index included in HH neighboring
index_s2 = verify_exclude_neighbouring(index_s2, index_s1_neigh)

In [ ]:
# print index_s4
index_s2

In [ ]:
# apply the function to verify and/o exclude any index included in HL neighboring
index_s2 = verify_exclude_neighbouring(index_s2, index_s4_neigh)

In [ ]:
# apply the function to get neighbours indexes of each items in index_s2  list
index_s2_neigh = get_neighbouring(index_s2)

In [ ]:
# apply the function to verify and/o exclude any index included in HH neighboring
index_s2_neigh = verify_exclude_neighbouring(index_s2_neigh, index_s1_neigh)

In [ ]:
# apply the function to verify and/o exclude any index included in HL neighboring
index_s2_neigh = verify_exclude_neighbouring(index_s2_neigh, index_s4_neigh)

In [ ]:
# get data of items in index_s2_neigh list
S2 = gdf_ca_rrrm_pj.loc[index_s2_neigh]

In [ ]:
# move geometry column to the end
S2 = S2[[col for col in S2.columns if col != 'geometry'] + ['geometry']]

In [ ]:
# Visualize S2 dataframe
init_notebook_mode(all_interactive=True)
S2

In [ ]:
# plot S2 neighbouring
plot_neighbouring(S2)

We observe a contiguous cluster consistent with the Queen contiguity matrix; however, we will exclude indices 110, 111 and 136 because they do not have statistically significant LISA p-values. We will also exclude index 135, as it is an island within the cluster

In [ ]:
# get list of indexes to drop
index_to_exclude_S2 = S2[S2['SIG_QUAD']=="NSC"].index.to_list()
# add index 135
#index_to_exclude_S2.append(135)

In [ ]:
# verify indexes to exclude
index_to_exclude_S2

In [ ]:
# exclude indexes from index_s2_neigh
index_s2_neigh = [item for item in index_s2_neigh if item not in index_to_exclude_S2]

In [ ]:
# print index_s2_neigh
index_s2_neigh

In [ ]:
# reselect data and visualize the datafarme
S2 = gdf_ca_rrrm_pj.loc[index_s2_neigh]
S2 = S2[[col for col in S2.columns if col != 'geometry'] + ['geometry']]
init_notebook_mode(all_interactive=True)
S2

In [ ]:
# Replot S2 neighbouring
plot_neighbouring(S2)

In [ ]:
# set cluster class in gdf_ca_rrrm_pj dataset (CLUSTERS == LH)
for index in index_s2_neigh:
  gdf_ca_rrrm_pj.loc[index, 'CLUSTERS'] = 'LH'

In [ ]:
#  move geometry column to the end
gdf_ca_rrrm_pj = gdf_ca_rrrm_pj[[col for col in gdf_ca_rrrm_pj.columns if col != 'geometry'] + ['geometry']]

In [ ]:
# visiualize rows of LH cluster
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm_pj.loc[index_s2_neigh]

##### Mapping LH cluster

In [ ]:
LH_data = gdf_ca_rrrm_pj[gdf_ca_rrrm_pj['CLUSTERS'] == 'LH']
non_LH_data = gdf_ca_rrrm_pj[gdf_ca_rrrm_pj['CLUSTERS'] != 'LH']

# Set up figure and subplots (Unpack axes correctly as a list/array)
fig, axes = plt.subplots(1, 2, figsize=(14, 7), gridspec_kw={'wspace': 0.05})
ax1 = axes[0]
ax2 = axes[1]

# --- First Subplot (Map 1) ---
non_LH_data.plot(ax=ax1, color='none', edgecolor='black', linewidth=0.15)
gdf_nan_pj.plot(ax=ax1, color='none', edgecolor='black', linewidth=0.15)
LH_data.plot(ax=ax1, color='#fdae61', edgecolor='black', linewidth=0.3)
ax1.set_axis_off()

# --- Second Subplot (Map 2) ---
LH_color = '#abd9e9'
LH_data.plot(ax=ax2, color=LH_color, edgecolor='black', linewidth=0.5)

# Collect text objects
texts = []
for idx, row in LH_data.iterrows():
    centroid = row.geometry.centroid
    val_str = str(row['IDDPTO'])

    # Optional Manual Hinting: Provide a tiny initial nudge to tight clusters
    # so adjust_text knows which direction to push them out smoothly.
    x_offset, y_offset = 0, 0
    if val_str in ['70042', '50021']:
        x_offset = -1500
    elif val_str in ['70098', '50070']:
        x_offset = 1500

    t = ax2.text(
        centroid.x + x_offset, centroid.y + y_offset, val_str,
        fontsize=7, color='black', weight='bold',
        ha='center', va='center'
    )
    texts.append(t)

# Force-adjust overlapping elements away from each other and the geometries
adjust_text(
    texts,
    ax=ax2,
    arrowprops=dict(arrowstyle="-", color='#555555', linewidth=0.6, alpha=0.7),
    expand_text=(1.4, 1.6),
    force_text=(0.4, 0.6),
    force_static=(0.2, 0.2),
    lim=2000
)

ax2.set_axis_off()

# --- Map Inset Connectors ---
if not LH_data.empty:
    minx, miny, maxx, maxy = LH_data.union_all().bounds
    rect_patch = mpatches.Rectangle(
        (minx, miny), maxx - minx, maxy - miny,
        facecolor='none', edgecolor='black', linewidth=1, linestyle='--', zorder=5
    )
    ax1.add_patch(rect_patch)

    # Arrow pointing from Map 1 to Map 2
    ax1.annotate(
        '', xy=(0, 0.62), xycoords=ax2.transAxes,
        xytext=(maxx, miny + (maxy - miny) / 2), textcoords=ax1.transData,
        arrowprops=dict(facecolor='black', edgecolor='black', width=0.1, headwidth=5, headlength=7)
    )

# --- General Title and Shared Legend ---
fig.suptitle(
    "Low predicted asthma mortality rate surrounded by \nHigh RR of respiratory mortality",
    fontsize=14, fontweight='bold', y=0.95
)

lh_patch = mpatches.Patch(facecolor=LH_color, edgecolor='black', linewidth=0.5, label='Low-High (LH) - "Spatial Outliers"')
fig.legend(
    handles=[lh_patch], loc='lower center',
    bbox_to_anchor=(0.5, 0.05), ncol=1, frameon=False, fontsize=10
)

plt.show()

In [ ]:
# display IDDPTO, PROV and DPTO LH cluster
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm_pj.loc[index_s2_neigh, ['IDDPTO', 'PROV', 'DPTO']]

#### LL Classification (Low predicted asthma mortality rate surrounded by Low RR of respiratory mortality)

In [ ]:
# get index of SIG_QUAD == S3 (significant LISA p-value in quadrant 3)
index_s3 = gdf_ca_rrrm_pj[gdf_ca_rrrm_pj['SIG_QUAD'] == "S3"].index

In [ ]:
# instex to list
index_s3 = index_s3.to_list()

In [ ]:
# print index
index_s3

In [ ]:
# apply the function to verify and/o exclude any index included in HH neighboring
index_s3 = verify_exclude_neighbouring(index_s3, index_s1_neigh)

In [ ]:
# apply the function to verify and/o exclude any index included in HL neighboring
index_s3 = verify_exclude_neighbouring(index_s3, index_s4_neigh)

In [ ]:
# apply the function to verify and/o exclude any index included in LH neighboring
index_s3 = verify_exclude_neighbouring(index_s3, index_s2_neigh)

In [ ]:
# apply the function to get neighbours indexes of each items in index_s3  list
index_s3_neigh = get_neighbouring(index_s3)

In [ ]:
# apply the function to verify and/o exclude any index included in HH neighboring
index_s3_neigh = verify_exclude_neighbouring(index_s3_neigh, index_s1_neigh)

In [ ]:
# apply the function to verify and/o exclude any index included in HL neighboring
index_s3_neigh = verify_exclude_neighbouring(index_s3_neigh, index_s4_neigh)

In [ ]:
# apply the function to verify and/o exclude any index included in LH neighboring
index_s3_neigh = verify_exclude_neighbouring(index_s3_neigh, index_s2_neigh)

In [ ]:
# get data of items in index_s3_neigh list
S3 = gdf_ca_rrrm_pj.loc[index_s3_neigh]

In [ ]:
# move geometry column to the end
S3 = S3[[col for col in S3.columns if col != 'geometry'] + ['geometry']]

In [ ]:
# Visualize S3 dataframe
init_notebook_mode(all_interactive=True)
S3

In [ ]:
# plot S3 neighbouring
plot_neighbouring(S3)

We observe two contiguous clusters consistent with the Queen contiguity matrix; however, we will exclude indices 75, 112, 114, and 120 because they do not have statistically significant LISA p-values. After removing these indices, we are left with two isolated departments (indices 86 and 118) that are not consistent with the Queen contiguity structure. Therefore, in this case, **no LL cluster is identified.**

In [ ]:
# save gdf_ca_rrrm_pj as geopackage
#gdf_ca_rrrm_pj.to_file('pdt/rr_respiratory_mortality/data/gpkg/gdf_ca_rrrm_pj_cluster.gpkg', driver='GPKG')

## 🌍 Mapping Bivariate LISA Clusters

In [ ]:
# HH = crimson  → high mortality, high neighbouring attributable rate
# LH = lightblue → low mortality, high neighbouring attributable rate
# HL = orange    → high mortality, low neighbouring attributable rate
# NO CLUSTER = lightgrey
cluster_colors = {
    'HH':'#d7191c',# crimson red
    'LH':'#4682B4',# light blue
    'HL':'#fdae61',# orange
    'NO CLUSTER': '#949494'# light grey
}

In [ ]:
# Map cluster labels to colours in the GeoDataFrame
# Every department row receives a hex colour based on its cluster label
gdf_ca_rrrm_pj['COLOR'] = gdf_ca_rrrm_pj['CLUSTERS'].map(cluster_colors)

In [ ]:
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm_pj

In [ ]:
# set up figure and axis
fig, ax = plt.subplots(1, 1, figsize=(5, 8))

# Plot the 170 analysed departments coloured by LISA cluster
# facecolor comes from the 'COLOR' column; edgecolor provides department borders
gdf_ca_rrrm_pj.plot(
    ax=ax,
    color=gdf_ca_rrrm_pj['COLOR'], # LISA cluster colour per department
    edgecolor='white',             # thin white border between departments
    linewidth=0.3
)

# Overlay NaN departments as hollow polygons with black border
gdf_nan_pj.plot(
    ax=ax,
    color='none',
    edgecolor='black',
    linewidth=0.15
)

# Add hollowed dashed rectangles around each cluster type
for cluster_type, color_code in cluster_colors.items():
    if cluster_type != 'NO CLUSTER':
        cluster_data = gdf_ca_rrrm_pj[gdf_ca_rrrm_pj['CLUSTERS'] == cluster_type]
        if not cluster_data.empty:
            minx, miny, maxx, maxy = cluster_data.union_all().bounds
            rect_patch = mpatches.Rectangle((minx, miny), maxx - minx, maxy - miny,
                                            facecolor='none', edgecolor=color_code, linewidth=1.5, linestyle='--', zorder=5)
            ax.add_patch(rect_patch)

# Build manual legend
# Each patch represents one cluster type with its corresponding colour
legend_labels = {
    #'HH — High / High (n={})'.format((gdf_ca_rrrm_pj['CLUSTERS']=='HH').sum()):'#d7191c',
    #'LL — Low / Low (n={})'.format((gdf_ca_rrrm_pj['CLUSTERS']=='LL').sum()):'#2c7bb6',
    'LH — Low / High (n={})'.format((gdf_ca_rrrm_pj['CLUSTERS']=='LH').sum()):'#4682B4',
    'HL — High / Low (n={})'.format((gdf_ca_rrrm_pj['CLUSTERS']=='HL').sum()):'#fdae61',
    'Not Significant (n={})'.format((gdf_ca_rrrm_pj['CLUSTERS']=='NO CLUSTER').sum()):'#949494',
    'No data': 'none'   # hollow patch for NaN depts
}

patches = []
for label, color in legend_labels.items():
    if color == 'none':
        # Use a patch with no fill but a black edge for NaN departments
        patch = mpatches.Patch(
            facecolor='none', edgecolor='black',
            linewidth=0.8, label=label
        )
    else:
        patch = mpatches.Patch(facecolor=color, edgecolor='white',
                               linewidth=0.25, label=label)
    patches.append(patch)

ax.legend(
    handles=patches,
    bbox_to_anchor= (1.2, 0.3),  # Adjust the legend's position
    fontsize=8,
    # title='Bivariate LISA Cluster\n(CA_2022_PRED \u00d7 AC_RATE_2022_C)',
    # title_fontsize=8,
    framealpha=0.9,
    edgecolor='grey',
)

# Title and cartographic elements
ax.set_title(
    'Bivariate LISA Cluster Map\n'
    'Predicted Asthma Mortality Rate vs RR of Respiratory Mortality \n'
    'Argentina \u2014 2022 (Queen Matrix, p < 0.05)',
    fontsize=11, fontweight='bold', pad=12
)

# Remove axis frame for a clean map appearance
ax.set_axis_off()

plt.tight_layout()
plt.show()

## 📚 References

[1] D. Wartenberg, “Multivariate Spatial Correlation: A Method for Exploratory Geographical Analysis,” Geographical Analysis, vol. 17, no. 4, pp. 263–283, 1985, doi: 10.1111/j.1538-4632.1985.tb00849.x.

[2] L. Zhang et al., “Socio-spatial inequalities in accessibility of Indigenous community-controlled mental health services in South East Queensland, Australia,” Int J Health Geogr, vol. 24, no. 1, p. 24, Sep. 2025, doi: 10.1186/s12942-025-00415-9.

[3] T.-H. Cao and T.-H. Dao, “Identification of spatial clusters and spatial outliers of HIV/AIDS using local Moran’s I statistic,” World Journal of Biology Pharmacy and Health Sciences, vol. 17, no. 1, pp. 101–108, 2024, doi: 10.30574/wjbphs.2024.17.1.0012.

[4] Y. Jia et al., “Spatial Epidemiological Analysis of Keshan Disease in China,” Ann Glob Health, vol. 88, no. 1, p. 79, doi: 10.5334/aogh.3836.

[5] A. A. Gwani, S. K. Sek, and A. A. Gwani, “A Systematic PRISMA Literature Survey on Renewable Energy Spatial Modelling and Proposed Advanced Statistical GIS-based Kth Order Queen Contiguity Methodology for Sustainable Development: Target Implementation to African Countries’ Clustering,” Mar. 06, 2024, Research Square. doi: 10.21203/rs.3.rs-4016566/v1.




